# Getting Started with Python: Exploratory Data Analysis (Polars)

**Author:** Alp Tezbasaran  
**Last updated:** 2026-02-09

This notebook is designed for a live workshop. It’s intentionally *hands-on* and *iterative*: we’ll ask questions, inspect the data, try a quick transformation or plot, then repeat.


## 0. Accessibility: theme, text size, and output

A few quick adjustments improve readability during live teaching.

- **Tools → Settings → Theme**: Light / Dark / System
- **Tools → Settings → Editor → Font size**: increase if needed
- **Browser zoom**: Cmd/Ctrl `+` or `-`

Teaching tip: agree on a standard zoom level at the start.


## Learning outcomes

By the end of this workshop, you will be able to:

- Load tabular data into Polars
- Profile data (shape, schema, missingness, unique values)
- Use **expressions** to filter and create features
- Summarize with `group_by().agg()` and explain what information is lost
- Use pivoting as **“groupby + aggregation + reshaping”**
- Make quick univariate and bivariate plots for exploration
- Recognize when to use **lazy** execution for performance


## The EDA loop (a simple roadmap)

EDA is not linear. A useful loop is:

1. **Inventory**: What columns do we have? What types? How big?
2. **Quality**: Missing values, duplicates, weird values
3. **Univariate**: What does each variable look like?
4. **Bivariate / multivariate**: What relationships appear?
5. **Feature ideas**: Create reinforces / clarifies relationships
6. **Summarize & communicate**: tables/plots + short written takeaways

We’ll follow that loop on a few small datasets.


## 1. Setup

We’ll use **Polars** for data work and **matplotlib** for plotting.

If you’re in Google Colab, install Polars once per session.


In [ ]:
# If you are using Colab, uncomment the next line
# !pip -q install polars

import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

# Data folder (relative to this notebook)
data_dir = Path('data')

# Keep plots readable in workshops
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True


## Polars: the 3 practical pillars

Polars queries are easiest to understand in three pieces:

1. **DataFrame / LazyFrame methods** define the pipeline *structure* (e.g., `select`, `filter`, `with_columns`, `group_by`, `join`).
2. **Expressions** (e.g., `pl.col(...)`, `pl.when(...).then(...).otherwise(...)`) define the *meaning* and the math.
3. **Lazy execution** (`scan_*` + `collect()`) lets Polars optimize the whole query (often much faster on large files).

We’ll mostly use **eager** DataFrames today, and show **lazy** as an optional pattern.


### Expression primer (5 minutes)

In Polars, you rarely write row-by-row loops. You write **column expressions**.

Common building blocks:
- `pl.col('x')` → refer to a column
- `.alias('new_name')` → rename an expression result
- Boolean logic: `&` (and), `|` (or), `~` (not)
- Conditional: `pl.when(cond).then(val).otherwise(val)`
- String/date helpers: `.str.*`, `.dt.*`


In [ ]:
expr_demo = pl.DataFrame({'x':[1,2,3,None], 's':[' A ','b','b',None]})
(
    expr_demo
    .with_columns([
        pl.col('x').fill_null(0).alias('x_filled'),
        pl.col('s').str.to_lowercase().str.strip_chars().fill_null('missing').alias('s_clean'),
        pl.when(pl.col('x').is_null()).then('was_null').otherwise('ok').alias('flag')
    ])
)


## 2. Load datasets

We’ll use three small datasets:

- `NCSU_Mascots_v1.csv` *(synthetic)*
- `NCSU Celebrity Graduates_v1.csv` *(synthetic)*
- `penguins.csv` *(public dataset, light cleaning demo)*

Even with synthetic data, practice good habits: avoid re-identification and think about bias.


In [ ]:
mascots = pl.read_csv(data_dir / 'NCSU_Mascots_v1.csv')
celebs = pl.read_csv(data_dir / 'NCSU Celebrity Graduates_v1.csv')
penguins_raw = pl.read_csv(data_dir / 'penguins.csv')

(mascots.shape, celebs.shape, penguins_raw.shape)


## 3. Inventory & profiling

Start every EDA with:
- **shape** (rows, columns)
- **schema** (column types)
- **preview** (head/tail)

Then a quick profile:
- missingness
- numeric summaries
- categorical cardinality (how many unique values?)


In [ ]:
print('Mascots:', mascots.shape)
print(mascots.schema)
mascots.head(5)


In [ ]:
print('Celebs:', celebs.shape)
print(celebs.schema)
celebs.head(5)


In [ ]:
print('Penguins:', penguins_raw.shape)
print(penguins_raw.schema)
penguins_raw.head(5)


### Missingness + basic numeric summary


In [ ]:
mascots.null_count()


In [ ]:
mascots.describe()


### Categorical “shape” (unique counts)

This is a fast way to find columns with many categories (IDs, free text, etc.).


In [ ]:
(
    mascots
    .select([
        pl.all().n_unique().alias('n_unique')
    ])
    .transpose(include_header=True, header_name='column', column_names=['n_unique'])
    .sort('n_unique', descending=True)
)


### Try it yourself

1) Which `mascots` columns have the most missing values?  
2) Which columns have the most unique values?

<details>
<summary>Hint</summary>
Use `.null_count()` and `.transpose()` to sort.
</details>


In [ ]:
# 1) Missingness ranking
# Your code here

# 2) Unique-value ranking
# Your code here


## 4. Univariate exploration

Univariate exploration answers: **“What does this variable look like by itself?”**

- Numeric: distribution (histogram), outliers (boxplot)
- Categorical: counts (bar chart)

We’ll do a quick categorical count on `mascots` and a histogram on `celebs` GPA.


In [ ]:
species_counts = (
    mascots
    .group_by('Species')
    .len()
    .sort('len', descending=True)
)
species_counts


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(species_counts['Species'].to_list(), species_counts['len'].to_list())
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.title('Mascots by Species')
plt.tight_layout()
plt.show()


In [ ]:
celebs_typed = celebs.with_columns([
    pl.col('GPA').cast(pl.Float64),
    pl.col('Workstudy Hourly Rate').cast(pl.Float64),
    pl.col('Loved Library?').cast(pl.Int64),
])

gpa = celebs_typed['GPA'].drop_nulls().to_list()
plt.figure(figsize=(6,4))
plt.hist(gpa, bins=12)
plt.xlabel('GPA')
plt.ylabel('Count')
plt.title('GPA distribution')
plt.tight_layout()
plt.show()


### Try it yourself

Make a bar chart of **counts by College** in the celebs dataset.

<details>
<summary>Solution</summary>

```python
college_counts = celebs.group_by('College').len().sort('len', descending=True)
plt.figure(figsize=(8,4))
plt.bar(college_counts['College'].to_list(), college_counts['len'].to_list())
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.title('Celebs by College')
plt.tight_layout()
plt.show()
```

</details>


In [ ]:
# Your code here


## 5. Cleaning + feature creation (Penguins)

Cleaning is easiest when it’s tied to a question.

Here we’ll:
- standardize the `sex` column
- drop rows missing key measurements (for a specific analysis)
- create a feature (`bill_ratio`) **before** we summarize

Note: dropping rows is not always appropriate; it depends on your goal.


In [ ]:
penguins = (
    penguins_raw
    .with_columns(
        pl.col('sex')
          .str.to_lowercase()
          .str.strip_chars()
          .alias('sex')
    )
    .with_columns(
        pl.when(pl.col('sex').is_null())
          .then('unknown')
          .otherwise(pl.col('sex'))
          .alias('sex')
    )
)

penguins.null_count()


In [ ]:
penguins_clean = penguins.drop_nulls(
    subset=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
).with_columns(
    (pl.col('bill_length_mm') / pl.col('bill_depth_mm')).alias('bill_ratio')
)

penguins_clean.head(5)


### Sanity check distributions (univariate, after cleaning)

A quick plot helps confirm we didn’t do something surprising.


In [ ]:
mass = penguins_clean['body_mass_g'].to_list()
plt.figure(figsize=(6,4))
plt.hist(mass, bins=15)
plt.xlabel('Body mass (g)')
plt.ylabel('Count')
plt.title('Penguins: body mass after dropping missing values')
plt.tight_layout()
plt.show()


## 6. Bivariate exploration

Bivariate exploration asks: **“How do two variables relate?”**

Common patterns:
- numeric ↔ numeric: scatter plot, correlation
- category ↔ numeric: boxplot, group summaries

We’ll look at bill measurements and body mass by species.


In [ ]:
x = penguins_clean['bill_length_mm'].to_list()
y = penguins_clean['bill_depth_mm'].to_list()
plt.figure(figsize=(6,4))
plt.scatter(x, y, s=10)
plt.xlabel('Bill length (mm)')
plt.ylabel('Bill depth (mm)')
plt.title('Penguins: bill length vs bill depth')
plt.tight_layout()
plt.show()


In [ ]:
penguins_clean.select([
    pl.corr('bill_length_mm', 'bill_depth_mm').alias('corr_bill_len_depth'),
    pl.corr('flipper_length_mm', 'body_mass_g').alias('corr_flipper_mass'),
])


In [ ]:
mass_by_species = (
    penguins_clean
    .group_by('species')
    .agg([
        pl.len().alias('n'),
        pl.col('body_mass_g').mean().alias('mean_mass'),
        pl.col('body_mass_g').median().alias('median_mass'),
        pl.col('body_mass_g').quantile(0.9).alias('p90_mass'),
    ])
    .sort('mean_mass', descending=True)
)
mass_by_species


### Try it yourself

1) Filter penguins to a single island and compute mean body mass.  
2) Make a scatter plot of flipper length vs body mass.

<details>
<summary>Hint</summary>
Use `.filter(pl.col('island') == 'Biscoe')` and `plt.scatter(...)`.
</details>


In [ ]:
# 1) Mean body mass on one island
# Your code here

# 2) Scatter: flipper_length_mm vs body_mass_g
# Your code here


## 7. Grouping + aggregation: choosing the grain

**Grouping** decides what one row in your output represents.

- `group_by(keys)` partitions rows into groups
- `.agg(...)` collapses each group (information is lost)

Rule of thumb: do row-level feature creation first, then group/aggregate last.


In [ ]:
summary = (
    celebs_typed
    .group_by(['College', 'Workstudy Position'])
    .agg([
        pl.len().alias('n'),
        pl.col('GPA').mean().alias('gpa_mean'),
        pl.col('GPA').median().alias('gpa_median'),
        pl.col('Workstudy Hourly Rate').mean().alias('rate_mean'),
    ])
    .sort(['College','n'], descending=[False, True])
)
summary.head(10)


### Window functions: group-aware summaries without losing rows

Sometimes you want group context **but you don’t want to collapse rows**.

Window functions use `.over(group_keys)`.


In [ ]:
celebs_with_context = (
    celebs_typed
    .with_columns([
        pl.col('GPA').mean().over('College').alias('college_gpa_mean'),
        (pl.col('GPA') - pl.col('GPA').mean().over('College')).alias('gpa_minus_college_mean'),
    ])
)
celebs_with_context.select(['Name','College','GPA','college_gpa_mean','gpa_minus_college_mean']).head(8)


## 8. Pivoting = groupby + aggregation + reshaping

A pivot table answers: **“Which values should become columns?”**

A pivot needs *one value per cell*. If multiple rows map to the same cell, you must choose an aggregation.

Here we pivot average GPA by College (rows) and Workstudy Position (columns).


In [ ]:
pivot_gpa = celebs_typed.pivot(
    values='GPA',
    index='College',
    columns='Workstudy Position',
    aggregate_function='mean'
)
pivot_gpa


### Try it yourself

Create a pivot of **mean hourly rate** by `College` (rows) and `Workstudy Position` (columns).

<details>
<summary>Solution</summary>

```python
celebs_typed.pivot(
    values='Workstudy Hourly Rate',
    index='College',
    columns='Workstudy Position',
    aggregate_function='mean'
)
```

</details>


In [ ]:
# Your code here


## 9. Joins (combining tables)

Joins combine columns from two tables using keys.

In real projects, joins often introduce missingness (unmatched keys) — so always inspect results.

Below is a tiny, self-contained example (not tied to the workshop datasets).


In [ ]:
left = pl.DataFrame({'id':[1,2,3], 'x':[10,20,30]})
right = pl.DataFrame({'id':[2,3,4], 'y':['a','b','c']})

left.join(right, on='id', how='left')


## 10. Lazy execution (optional but important)

Use lazy mode when:
- files are large
- you want to push filters into the scan
- you want Polars to optimize the whole query

Key idea: `scan_*` builds a plan; `collect()` executes.


In [ ]:
lazy_plan = (
    pl.scan_csv(data_dir / 'NCSU_Mascots_v1.csv')
      .filter(pl.col('Species').is_not_null())
      .group_by('Species')
      .agg(pl.len().alias('n'))
      .sort('n', descending=True)
)

lazy_plan  # shows a LazyFrame (a plan)


In [ ]:
lazy_plan.explain()


In [ ]:
lazy_plan.collect().head(10)


## Wrap-up

What you practiced:

- Profiling: shape, schema, missingness, unique counts
- Univariate + bivariate exploration with quick plots
- Feature creation **before** aggregation
- `group_by().agg()` summaries (lossy)
- Window functions (group context without collapsing)
- Pivoting as **groupby + aggregation + reshape**
- Optional: lazy execution and query plans

### Next steps

- Add more bivariate plots (e.g., by subgroup)
- Add checks for outliers and impossible values
- Write down 3–5 findings in plain language
- Decide what you’d do next: cleaning, modeling, or reporting
